# TODO:

Politics / Mentions: two promising wallets
0x9e8ecc4cb3c4e48f544cba2fbbb252a6a65e8db8	
0x0cb10c40b0776e9ee8cef970af85724654dda76c


# Wallet Strategy Selection

This stage now prioritizes **copy-trigger wallet selection** on the **full trade stream**
(BUY + SELL), with simple concentration controls and a transparent final score that is
computed immediately before selecting wallets.

Notes:
- Open-buy skill metrics are kept for diagnostics/legacy comparison.
- Production wallet selection is `wallet_cohorts['multi_filter']` and is exported to
  `selected_wallets_v2.parquet`.


In [1]:
%load_ext autoreload
%autoreload 2

import datetime
from pathlib import Path
import json

import numpy as np
import pandas as pd
import pyarrow.dataset as ds

np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

## Configuration

In [2]:
# ── configuration ─────────────────────────────────────────────────────────────
RECENCY_DAYS     = 90

PRICE_BUCKET_BINS   = [0.0, 0.1, 0.25, 0.4, 0.6, 0.75, 0.9, 1.0]
PRICE_BUCKET_LABELS = [
    "0.00-0.10", "0.10-0.25", "0.25-0.40", "0.40-0.60",
    "0.60-0.75", "0.75-0.90", "0.90-1.00",
]

TRADES_DIR    = Path("../../data/polygon_trades_processed")
WORKSPACE_DIR = Path("../../data/trade_signals_workspace_v2")
WORKSPACE_DIR.mkdir(parents=True, exist_ok=True)

dataset = ds.dataset(TRADES_DIR, format="parquet")
DATASET_COLUMNS        = set(dataset.schema.names)
TRIGGER_TX_HASH_COLUMN = (
    "transaction_hash" if "transaction_hash" in DATASET_COLUMNS
    else ("tx_hash" if "tx_hash" in DATASET_COLUMNS else None)
)

METRICS_A_PATH          = WORKSPACE_DIR / "wallet_metrics_train_a.parquet"
METRICS_B_PATH          = WORKSPACE_DIR / "wallet_metrics_train_b.parquet"
METRICS_FULL_PATH       = WORKSPACE_DIR / "wallet_metrics_full_train.parquet"
METRICS_TEST_PATH       = WORKSPACE_DIR / "wallet_metrics_test.parquet"
CALIBRATION_SIGNALS_PATH = WORKSPACE_DIR / "signal_events_train_b.parquet"
TEST_SIGNALS_PATH       = WORKSPACE_DIR / "signal_events_test.parquet"


## Derive train/test dates from data

In [3]:
# ── derive train/test boundary from is_train flag ───────────────────────────
_sample = pd.read_parquet(sorted(TRADES_DIR.glob("*.parquet"))[0], columns=["dt", "is_train"])
_sample["dt"] = pd.to_datetime(_sample["dt"], utc=True)
END_DATE_TRAIN   = _sample.loc[_sample["is_train"], "dt"].max().date()
TRAIN_START_DATE = _sample.loc[_sample["is_train"], "dt"].min().date()
# Split train_a / train_b at the trade-count median of training rows so that
# both halves always contain roughly equal numbers of trades even when the
# data is unevenly distributed over calendar time.
_train_rows = _sample.loc[_sample["is_train"]].sort_values("dt")
TRAIN_A_END_DATE = _train_rows["dt"].quantile(0.5, interpolation="nearest").date()
del _sample, _train_rows
print(f"END_DATE_TRAIN={END_DATE_TRAIN}  TRAIN_A_END_DATE={TRAIN_A_END_DATE}")


END_DATE_TRAIN=2026-04-15  TRAIN_A_END_DATE=2026-01-06


## Compute / load wallet skill metrics (legacy diagnostics)

These open-buy metrics are not used for the final copy-trigger wallet selection.


In [4]:
from polymarket_analysis.wallet_selection.metrics import compute_wallet_skill_workspace

if all(p.exists() for p in [METRICS_A_PATH, METRICS_B_PATH, METRICS_FULL_PATH, METRICS_TEST_PATH]):
    train_a_metrics    = pd.read_parquet(METRICS_A_PATH)
    train_b_metrics    = pd.read_parquet(METRICS_B_PATH)
    full_train_metrics = pd.read_parquet(METRICS_FULL_PATH)
    test_metrics       = pd.read_parquet(METRICS_TEST_PATH)
else:
    train_a_metrics, train_b_metrics, full_train_metrics, test_metrics = (
        compute_wallet_skill_workspace(
            dataset,
            end_date_train=END_DATE_TRAIN,
            train_a_end_date=TRAIN_A_END_DATE,
            recency_days=RECENCY_DAYS,
        )
    )
    train_a_metrics.to_parquet(METRICS_A_PATH, index=False)
    train_b_metrics.to_parquet(METRICS_B_PATH, index=False)
    full_train_metrics.to_parquet(METRICS_FULL_PATH, index=False)
    test_metrics.to_parquet(METRICS_TEST_PATH, index=False)

pd.Series({
    "train_a_wallets":    len(train_a_metrics),
    "train_b_wallets":    len(train_b_metrics),
    "full_train_wallets": len(full_train_metrics),
    "test_wallets":       len(test_metrics),
}).to_frame("value")


,value
train_a_wallets,10964
train_b_wallets,10964
full_train_wallets,10964
test_wallets,10964


## Cohort selection sweep (skill path)

Grid-search over metrics × top-N using train-a → train-b persistence.

In [5]:
from polymarket_analysis.wallet_selection.selector import (
    CANDIDATE_METRICS,
    cohort_selection_sweep,
    select_wallets,
)

cohort_sweep = cohort_selection_sweep(train_a_metrics, train_b_metrics, CANDIDATE_METRICS)
cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge"], ascending=False
).head(20)


,metric,top_n,wallets_found_in_train_b,train_b_open_buy_trades,train_b_weighted_prob_edge,train_b_avg_prob_edge,train_b_avg_copy_roi_capped,train_b_total_copy_pnl_usdc,train_b_hit_rate
10,avg_copy_roi_capped,50,50,1813,0.07,0.12,1.52,103284.02,0.42
11,avg_copy_roi_capped,100,100,2695,0.07,0.10,1.12,143225.93,0.42
0,prob_edge_shrunk,50,50,4204,0.01,0.12,0.93,115130.45,0.47
22,roi_sharpe,200,200,3250,0.01,0.07,0.85,101101.91,0.70
12,avg_copy_roi_capped,200,200,5272,0.06,0.11,0.82,248959.70,0.48
1,prob_edge_shrunk,100,100,5418,0.03,0.11,0.78,201409.43,0.51
13,avg_copy_roi_capped,300,300,7046,0.06,0.07,0.76,384078.84,0.50
23,roi_sharpe,300,300,4467,0.01,0.07,0.70,117444.18,0.70
6,weighted_prob_edge_shrunk,100,100,2399,0.06,0.09,0.68,256835.63,0.47
21,roi_sharpe,100,100,2473,0.01,0.03,0.66,73063.49,0.78


In [6]:
# pick best metric / top-N
best_row = cohort_sweep.sort_values(
    ["train_b_avg_copy_roi_capped", "train_b_weighted_prob_edge", "train_b_open_buy_trades"],
    ascending=[False, False, False],
).iloc[0]
BEST_SELECTION_METRIC = best_row["metric"]
BEST_TOP_N            = int(best_row["top_n"])
{"best_metric": BEST_SELECTION_METRIC, "best_top_n": BEST_TOP_N}


{'best_metric': 'avg_copy_roi_capped', 'best_top_n': 50}

## Select wallets (full-stream copy-trigger path)


## Build wallet profiles and signal events

In [7]:
# from polymarket_analysis.signal.builder import (
#     build_wallet_profiles,
#     build_signal_events,
# )

# selected_wallet_profiles = build_wallet_profiles(
#     dataset, selected_wallets, period="full_train",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE
# )
# selected_wallet_profiles.to_parquet(
#     WORKSPACE_DIR / "selected_wallet_profiles_v2.parquet", index=False
# )

# # Force-regenerate signal caches: the schema now includes all event types
# # (open_buy, add_buy, close_sell, reduce_sell) plus copy_price/copy_market_key/
# # copy_token_winner columns. Delete stale parquets so they are always rebuilt.
# CALIBRATION_SIGNALS_PATH.unlink(missing_ok=True)
# TEST_SIGNALS_PATH.unlink(missing_ok=True)

# _, train_b_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="train_b",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# train_b_signals.to_parquet(CALIBRATION_SIGNALS_PATH, index=False)

# _, test_signals = build_signal_events(
#     dataset, selected_wallet_profiles, period="test",
#     end_date_train=END_DATE_TRAIN, train_a_end_date=TRAIN_A_END_DATE,
#     price_bucket_bins=PRICE_BUCKET_BINS, price_bucket_labels=PRICE_BUCKET_LABELS,
#     tx_hash_column=TRIGGER_TX_HASH_COLUMN,
# )
# test_signals.to_parquet(TEST_SIGNALS_PATH, index=False)

# print(f"train_b signals: {len(train_b_signals):,}  test signals: {len(test_signals):,}")
# print(f"event types: {train_b_signals['event_type'].value_counts().to_dict()}")

## Calibrate signal scoring on train-B

In [8]:
# from polymarket_analysis.signal.scorer import (
#     build_calibration_tables,
#     apply_signal_score,
#     select_signal_threshold,
# )

# # Calibration tables are built from open_buy events only — the scoring model
# # is designed for open-buy conviction features. Non-open-buy rows get neutral
# # defaults and will not be used to calibrate the scorer.
# train_b_open_buys = train_b_signals[train_b_signals["event_type"] == "open_buy"].copy()

# price_table, consensus_table, global_edge = build_calibration_tables(train_b_open_buys)
# train_b_scored = apply_signal_score(train_b_open_buys, price_table, consensus_table)
# SIGNAL_THRESHOLD = select_signal_threshold(train_b_scored)
# print(f"Global edge: {global_edge:.4f}")
# print(f"Selected signal threshold: {SIGNAL_THRESHOLD:.2f}")

# # score distribution
# score_deciles = (
#     train_b_scored.assign(
#         score_decile=lambda df: pd.qcut(df["signal_score"], q=10, labels=False, duplicates="drop")
#     )
#     .groupby("score_decile")
#     .agg(
#         count=("signal_score", "size"),
#         avg_signal_score=("signal_score", "mean"),
#         avg_copy_roi_capped=("copy_roi_capped", "mean"),
#     )
#     .reset_index()
# )
# score_deciles

## Build wallet cohorts (skill path)

In [9]:
from polymarket_analysis.wallet_selection.selector import build_wallet_cohorts

# wallet_cohorts = build_wallet_cohorts(
#     full_train_metrics, train_b_open_buys, selected_wallets,
#     top_n=BEST_TOP_N,
# )
# {name: len(df) for name, df in wallet_cohorts.items()}


## Full-stream wallet metrics for copy-trigger selection

Build per-wallet metrics on the full training trade stream (BUY + SELL).
Copyability is considered here (stage1), not in stage0 filtering.


In [10]:
from polymarket_analysis.wallet_selection.volatility import (
    compute_wallet_metrics,
    filter_wallets_by_volatility,
)

from polymarket_analysis.data.data_catalogue import load_markets_processed
mdf = load_markets_processed()
mdf = mdf[
    ~(mdf['primary_tag'].isin(['Sports', 'Crypto']))
    & (mdf['winner_token_id'].notna())
].copy().reset_index(drop=True)

# Load full processed trades for the full-stream path
trade_files = sorted(TRADES_DIR.glob("*.parquet"))
df_full = pd.concat([pd.read_parquet(f) for f in trade_files], ignore_index=True)

df_full = df_full.merge(mdf, on="condition_id", how="inner")

df_full['outcome'] = df_full['outcome_x']
del df_full['outcome_x'], df_full['outcome_y']

df_full["dt"] = pd.to_datetime(df_full["dt"], utc=True)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_full.columns and "quantity" not in df_full.columns:
    df_full = df_full.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_full["usdc_amount"] = df_full["usdc_amount"].astype(float)
df_full["final_value_usdc"] = df_full["final_value_usdc"].astype(float)
df_full["quantity"] = df_full["quantity"].astype(float)

# Full-stream PnL / notional across BUY + SELL
df_full["pnl"] = np.where(
    df_full["side"] == "BUY",
    df_full["final_value_usdc"] - df_full["usdc_amount"],
    df_full["usdc_amount"] - df_full["final_value_usdc"],
)
df_full["notional"] = np.where(
    df_full["side"] == "BUY",
    df_full["usdc_amount"],
    df_full["quantity"] * (1 - df_full["price"].astype(float)),
)

df_slice = df_full[df_full["is_train"]].copy()
wallet_vol_train, _ = compute_wallet_metrics(df_slice)
print(len(wallet_vol_train))


37411


In [11]:
wallet_vol_train['copyable_pnl_factor'] = np.clip(
    wallet_vol_train['copyable_pnl'] / wallet_vol_train['total_pnl'].replace(0, np.nan),
    0,
    1.0,
).fillna(0.0)
wallet_vol_train['copyable_roi'] = wallet_vol_train['average_roi'] * wallet_vol_train['copyable_pnl_factor']

# Keep sorting deterministic for downstream filtering/inspection.
wallet_vol_train.sort_values('copyable_roi', ascending=False, inplace=True)
wallet_vol_train.reset_index(drop=True, inplace=True)


In [12]:
wallet_cohorts = {}

In [13]:
print(len(wallet_vol_train))
wallet_vol_train.head()


37411


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,...,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi
0,0xa231231e1e0a4d69fd7e51c3c6e6676b3f692e37,13.70,14,4,514.16,124.69,321.45,1.03,1.00,0.49,...,165.67,2025-12-22 15:47:30+00:00,118.12,307.14,2.46,0.20,0.00,0.24,1.00,118.12
1,0x4b1d2d479f67a33141d67005acf069e10a08bdfb,NaN,1,1,1.52,150.48,150.48,1.00,1.00,1.00,...,99.00,2026-02-27 06:10:00+00:00,99.00,0.00,0.00,0.00,0.00,99.00,1.00,99.00
2,0x8da0d29ec66508950ad72846ac3164db7cc41c53,NaN,1,1,2.50,205.83,205.83,1.00,1.00,1.00,...,82.33,2026-04-03 19:40:00+00:00,82.33,0.00,0.00,0.00,0.00,82.33,1.00,82.33
3,0xcbf45e4a1de6f54869fedf2ab7c064f248948723,1.72,10,4,643.81,1958.70,1762.99,1.00,1.00,1.00,...,82.33,2026-04-04 13:12:30+00:00,59.22,0.00,0.00,0.00,0.00,3.04,0.90,53.30
4,0x69f8c4434bf303a65141db9e3a92862d334a08c1,NaN,1,1,2.09,114.02,108.02,1.00,1.00,1.00,...,54.56,2026-04-02 09:05:00+00:00,54.56,0.00,0.00,0.00,0.00,54.56,0.95,51.68


In [14]:
# filtered_wallets_vol = filter_wallets_by_volatility(
#     wallet_vol_train,
#     min_buckets=20,
#     max_top5_pnl_pct=.6,
#     max_top_market_pnl_pct=0.5,
# )

# filtered_wallets_vol = wallet_vol_train.copy()

# filtered_wallets_vol = (
#     filtered_wallets_vol[
#         (filtered_wallets_vol['average_roi'] >= 0.04)
#         & (filtered_wallets_vol['0x0a'] / filtered_wallets_vol['total_pnl'] >= 0.5)
# ].sort_values("pnl_volatility", ascending=True).head(100)
# )

# print(f"Volatility-filtered wallets: {len(filtered_wallets_vol)}")

# # Build wallet_quality based on total_pnl rank (higher = better)
# filtered_wallets_vol = filtered_wallets_vol.copy().reset_index(drop=True)
# filtered_wallets_vol["wallet_quality"] = filtered_wallets_vol["total_pnl"].rank(
#     method="first", pct=True
# )

# Add as additional cohort (uses only train-b signals for trigger calibration)
# We intersect with wallets that have signals to ensure coverage
# vol_wallets_with_signals = set(train_b_signals["wallet"]).intersection(
#     set(filtered_wallets_vol["wallet"])
# )
# [
#     filtered_wallets_vol["wallet"].isin(vol_wallets_with_signals)
# ][["wallet", "wallet_quality"]].copy().reset_index(drop=True)

In [369]:
# Final wallet selection for copy-triggers: simple filters + absolute scores
TARGET_WALLET_COUNT = 120
MIN_WALLET_COUNT = 100

candidates = wallet_vol_train.copy()

# Backward-compatible defaults when opening older cached tables.
for col, default in {
    'top10_pnl_pct': np.nan,
    'top_market_abs_pnl_pct': np.nan,
    'market_pnl_hhi': np.nan,
    'positive_bucket_share': np.nan,
}.items():
    if col not in candidates.columns:
        candidates[col] = default

for col in [
    'average_roi', 'median_roi', 'num_buckets', 'num_markets',
    'pnl_volatility', 'max_drawdown_to_pnl',
    'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
    'copyable_roi', 'copyable_pnl_factor', 'copyable_pnl', 'total_notional',
    'max_copyable_drawdown_to_copyable_pnl', 'worst5_pnl_pct', 'positive_bucket_share',
]:
    if col in candidates.columns:
        candidates[col] = pd.to_numeric(candidates[col], errors='coerce')

# Hard filters (simple concentration controls + basic quality gates)
base_mask = (
    (candidates['average_roi'] >= 0.03)
    & (candidates['median_roi'] >= 0)
    & (candidates['num_buckets'] >= 20)
    & (candidates['num_markets'] >= 15)
    & (candidates['max_drawdown_to_pnl'] <= 0.30)
    & (candidates['top_market_pnl_pct'] < 0.55)
    & (candidates['top5_pnl_pct'] < 0.75)
    & (candidates['top10_pnl_pct'].fillna(0.85) < 0.90)
    & (candidates['top_market_abs_pnl_pct'].fillna(candidates['top_market_pnl_pct']) < 0.45)
    & (candidates['market_pnl_hhi'].fillna(0.20) < 0.20)
    & (candidates['median_dt'].dt.date <= (END_DATE_TRAIN - pd.Timedelta(days=60)))
    & (candidates['total_notional'] >= 5_000)
)

eligible = candidates[base_mask].copy()

# Safety fallback to keep enough signal flow.
if len(eligible) < MIN_WALLET_COUNT:
    relaxed_mask = (
        (candidates['num_buckets'] >= 15)
        & (candidates['num_markets'] >= 10)
        & (candidates['max_drawdown_to_pnl'] <= 0.40)
        & (candidates['top_market_pnl_pct'] < 0.65)
        & (candidates['top5_pnl_pct'] < 0.85)
        & (candidates['total_notional'] >= 2_500)
        & (candidates['median_dt'].dt.date <= (END_DATE_TRAIN - pd.Timedelta(days=30)))
    )
    eligible = candidates[relaxed_mask].copy()

if eligible.empty:
    raise ValueError('No wallets passed full-stream filters; relax constraints.')

# Absolute sample-strength multiplier (no percentile ranking).
eligible['sample_mult'] = np.clip(
    np.log1p(eligible['num_buckets']) / np.log(2000.0),
    0.20,
    1.00,
)

# --- Predictability score (distinct from raw PnL size) ---
eligible['downside_tail'] = eligible['worst5_pnl_pct'].abs().fillna(0.0)
eligible['predictability_score'] = eligible['sample_mult'] * (
    2.5 * eligible['average_roi'].fillna(0.0)
    + 1.5 * eligible['median_roi'].fillna(0.0)
    + 1.2 * (eligible['positive_bucket_share'].fillna(0.5) - 0.5)
    - 1.0 * eligible['pnl_volatility'].fillna(0.0)
    - 1.2 * eligible['max_drawdown_to_pnl'].fillna(0.0)
    - 0.8 * eligible['top_market_abs_pnl_pct'].fillna(eligible['top_market_pnl_pct']).fillna(0.0)
    - 0.6 * eligible['market_pnl_hhi'].fillna(0.0)
    - 0.5 * eligible['downside_tail']
)

# --- Copyable score (copy-trader realizability) ---
eligible['copyable_efficiency'] = eligible['copyable_pnl'].fillna(0.0) / (eligible['total_notional'].fillna(0.0) + 1.0)
eligible['copyable_dd_ratio'] = (
    eligible['max_copyable_drawdown_to_copyable_pnl']
    .fillna(eligible['max_drawdown_to_pnl'])
    .fillna(0.0)
)
eligible['copyable_score'] = (
    1.8 * eligible['copyable_roi'].fillna(0.0)
    + 1.2 * eligible['copyable_pnl_factor'].fillna(0.0)
    + 25.0 * eligible['copyable_efficiency'].clip(lower=-1.0, upper=1.0)
    - 0.8 * eligible['copyable_dd_ratio'].clip(lower=0.0)
    - 0.5 * eligible['top_market_abs_pnl_pct'].fillna(eligible['top_market_pnl_pct']).fillna(0.0)
)

# Final absolute score used for selection.
eligible['final_score'] = (
    0.60 * eligible['predictability_score']
    + 0.40 * eligible['copyable_score']
)

eligible = eligible[eligible['final_score'] > 0].copy()

eligible = eligible.sort_values('final_score', ascending=False).reset_index(drop=True)
selected_n = min(len(eligible), max(MIN_WALLET_COUNT, TARGET_WALLET_COUNT))
wallet_cohorts['multi_filter'] = eligible.head(selected_n).copy().reset_index(drop=True)
wallet_cohorts['multi_filter']['wallet_quality'] = wallet_cohorts['multi_filter']['final_score']

print(f"Eligible wallets after filters: {len(eligible):,}")
print(f"Selected wallets (multi_filter): {len(wallet_cohorts['multi_filter']):,}")
wallet_cohorts['multi_filter'].head(200)
# [[
#     'wallet', 'final_score', 'predictability_score', 'copyable_score',
#     'average_roi', 'median_roi', 'copyable_roi', 'copyable_pnl_factor',
#     'num_buckets', 'num_markets',
#     'top_market_pnl_pct', 'top_market_abs_pnl_pct', 'top5_pnl_pct', 'top10_pnl_pct', 'market_pnl_hhi',
# ]].head(200)


Eligible wallets after filters: 22
Selected wallets (multi_filter): 22


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
0,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,1.06,247,82,358092.33,1596.22,459.84,0.50,0.73,-0.20,0.27,0.25,0.11,0.78,0.00,2025-02-03 14:20:00+00:00,3.76,141.35,0.09,99.50,0.22,0.00,0.29,1.08,0.73,0.20,5.94,0.00,0.22,2.03,4.38,4.38
1,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,3.66,1287,195,86665.60,42409.16,6837.74,0.22,0.38,-0.23,0.31,0.18,0.06,0.55,0.11,2026-02-13 11:15:00+00:00,3.40,7373.80,0.17,5604.27,0.82,0.49,0.16,0.55,0.94,0.23,4.32,0.08,0.82,2.41,3.55,3.55
2,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.99,56,34,5978.58,1037.16,644.30,0.54,0.79,-0.27,0.14,0.10,0.05,0.72,0.18,2025-06-24 10:30:00+00:00,0.91,107.46,0.10,73.91,0.11,0.17,0.62,0.57,0.53,0.27,0.77,0.11,0.11,4.31,2.19,2.19
3,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.36,221,47,50819.14,6107.90,1358.21,0.40,0.62,-0.21,0.14,0.12,0.07,0.77,0.12,2025-02-23 16:25:00+00:00,1.07,631.30,0.10,449.21,0.33,0.12,0.22,0.24,0.71,0.21,1.03,0.03,0.33,1.04,1.04,1.04
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.04,5472,1784,299213.40,46376.75,6731.76,0.14,0.22,-0.06,0.06,0.03,0.00,0.51,0.01,2025-11-07 17:15:00+00:00,0.89,1030.77,0.02,1082.54,0.16,0.15,0.15,0.13,1.00,0.06,1.13,0.02,0.16,0.83,1.01,1.01
5,0x672225e5e1aba1c970ac613efd1505f4b7a10762,1.46,1113,319,255194.40,4860.34,1381.57,0.42,0.56,-0.03,0.13,0.13,0.05,0.88,0.00,2026-01-28 21:25:00+00:00,0.81,160.00,0.03,100.36,0.07,0.02,0.28,0.23,0.92,0.03,0.77,0.01,0.07,0.77,0.77,0.77
6,0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,0.59,1647,658,39055.76,2022.02,256.77,0.27,0.44,-0.28,0.07,0.06,0.01,0.65,0.05,2025-10-29 18:50:00+00:00,0.74,407.05,0.20,74.49,0.29,0.05,0.13,0.09,0.97,0.28,1.05,0.01,0.29,0.22,0.72,0.72
7,0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,0.52,3178,1105,82151.41,13233.71,486.75,0.19,0.29,-0.09,0.06,0.03,0.00,0.55,0.03,2025-09-30 18:15:00+00:00,0.80,613.32,0.05,404.04,0.83,0.16,0.04,0.03,1.00,0.09,1.45,0.01,0.83,-0.43,0.70,0.70
8,0x9783178832982d420baa733d2ec29b020eb9264f,0.77,53,26,8227.49,465.19,380.91,0.50,0.72,-0.14,0.16,0.15,0.08,0.95,0.04,2026-01-18 10:00:00+00:00,0.04,58.13,0.12,19.62,0.05,0.06,0.82,0.03,0.52,0.14,-0.25,0.05,0.05,2.07,0.68,0.68
9,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,1.52,124,67,13154.70,2148.21,1169.65,0.43,0.68,-0.34,0.16,0.08,0.03,0.53,0.06,2026-01-26 13:37:30+00:00,0.15,250.83,0.12,107.42,0.09,0.16,0.54,0.08,0.64,0.34,-0.89,0.09,0.09,2.92,0.63,0.63


In [367]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
wallet_cohorts = {}

In [371]:
filter_wallets = set(wallet_cohorts['multi_filter']["wallet"])
df_test =  df_full[
    (df_full['dt'] > pd.to_datetime("2026-04-16", utc=True))
    # & (df_full['wallet'] == '0xbacd00c9080a82ded56f504ee8810af732b0ab35')
    & (df_full['wallet'].isin(filter_wallets))
    ].groupby(['question', 'condition_id', 'outcome']).agg(
    # ].groupby(['wallet']).agg(
        pnl_sum=('pnl', 'sum'),
        trade_count=('pnl', 'size'),
        wallet_count=('wallet', 'nunique'),
    ).reset_index().sort_values('pnl_sum', key=abs, ascending=False)

big_conditions = df_test.groupby('condition_id').agg(
    abs_pnl_sum=('pnl_sum', lambda x: abs(x).sum()),
    min_abs_pnl_pct=('pnl_sum', lambda x: abs(x).min() / abs(x).sum() if abs(x).sum() > 0 else 0)
).reset_index().sort_values('abs_pnl_sum', ascending=False)



print(df_test['pnl_sum'].sum())
df_test = df_test[df_test['condition_id'].isin(big_conditions['condition_id'])].sort_values('question', ascending=False).merge(big_conditions, on='condition_id', how='left')

print(df_test[df_test['min_abs_pnl_pct']<0.1]['pnl_sum'].sum()) 

df_test[df_test['min_abs_pnl_pct'] < 0.1].sort_values(['abs_pnl_sum'], ascending=False).head(20)

23509.536958
8330.862507


,question,condition_id,outcome,pnl_sum,trade_count,wallet_count,abs_pnl_sum,min_abs_pnl_pct
1216,Strait of Hormuz traffic returns to normal by end of April?,0x924a2942747dd75703321a7c8d809c68f6a514c3b0f2a2e64274e02310634669,No,7629.71,174,1,7647.67,0.00
1217,Strait of Hormuz traffic returns to normal by end of April?,0x924a2942747dd75703321a7c8d809c68f6a514c3b0f2a2e64274e02310634669,Yes,17.96,14,2,7647.67,0.00
1261,Iran closes its airspace by May 24?,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,No,2621.84,31,2,2629.84,0.00
1262,Iran closes its airspace by May 24?,0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,Yes,8.00,3,1,2629.84,0.00
1148,US-Iran nuclear deal by April 30?,0xd08544f6162283dc8d0a82f16362aab837a8537379df9bbe604960eec9cd4618,Yes,-2161.57,49,1,2185.97,0.01
1149,US-Iran nuclear deal by April 30?,0xd08544f6162283dc8d0a82f16362aab837a8537379df9bbe604960eec9cd4618,No,24.40,1,1,2185.97,0.01
1208,Trump kiss by May 31?,0xcef981d46a1039b6ae02f578d2208302a8a3d63465d24363a1d65a86835a1ae8,Yes,-12.17,10,1,1431.46,0.01
1207,Trump kiss by May 31?,0xcef981d46a1039b6ae02f578d2208302a8a3d63465d24363a1d65a86835a1ae8,No,-1419.29,33,1,1431.46,0.01
580,"Will Trump say ""Make America Great Again"" this week?",0xbb56bad20130f9b79ddbf302a19ade676f1539a00f6c508d8077bcef779c11ab,No,885.40,12,2,886.51,0.00
584,"Will Trump say ""Make America Great Again"" this week?",0xbb56bad20130f9b79ddbf302a19ade676f1539a00f6c508d8077bcef779c11ab,Yes,-1.11,1,1,886.51,0.00


In [372]:
wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False).head(100)

,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
20,0x22dbd51e1a4e10fff072fa24300238c64033190f,0.70,10418,2573,702549.39,44426.92,11371.92,0.19,0.32,-0.11,0.09,0.03,0.00,0.67,0.08,2025-09-15 01:55:00+00:00,0.11,3689.31,0.08,2631.75,0.23,0.06,0.26,0.03,1.00,0.11,-0.28,0.02,0.23,0.57,0.06,0.06
1,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,3.66,1287,195,86665.60,42409.16,6837.74,0.22,0.38,-0.23,0.31,0.18,0.06,0.55,0.11,2026-02-13 11:15:00+00:00,3.40,7373.80,0.17,5604.27,0.82,0.49,0.16,0.55,0.94,0.23,4.32,0.08,0.82,2.41,3.55,3.55
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.04,5472,1784,299213.40,46376.75,6731.76,0.14,0.22,-0.06,0.06,0.03,0.00,0.51,0.01,2025-11-07 17:15:00+00:00,0.89,1030.77,0.02,1082.54,0.16,0.15,0.15,0.13,1.00,0.06,1.13,0.02,0.16,0.83,1.01,1.01
13,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,0.92,3812,1106,225426.08,29006.48,4236.02,0.20,0.32,-0.05,0.11,0.06,0.01,0.61,0.02,2025-10-13 11:25:00+00:00,0.45,745.47,0.03,555.95,0.13,0.13,0.15,0.07,1.00,0.05,0.25,0.02,0.13,0.63,0.40,0.40
5,0x672225e5e1aba1c970ac613efd1505f4b7a10762,1.46,1113,319,255194.40,4860.34,1381.57,0.42,0.56,-0.03,0.13,0.13,0.05,0.88,0.00,2026-01-28 21:25:00+00:00,0.81,160.00,0.03,100.36,0.07,0.02,0.28,0.23,0.92,0.03,0.77,0.01,0.07,0.77,0.77,0.77
3,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.36,221,47,50819.14,6107.90,1358.21,0.40,0.62,-0.21,0.14,0.12,0.07,0.77,0.12,2025-02-23 16:25:00+00:00,1.07,631.30,0.10,449.21,0.33,0.12,0.22,0.24,0.71,0.21,1.03,0.03,0.33,1.04,1.04,1.04
9,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,1.52,124,67,13154.70,2148.21,1169.65,0.43,0.68,-0.34,0.16,0.08,0.03,0.53,0.06,2026-01-26 13:37:30+00:00,0.15,250.83,0.12,107.42,0.09,0.16,0.54,0.08,0.64,0.34,-0.89,0.09,0.09,2.92,0.63,0.63
15,0x444e1b2ccf56b63e6b54e1357df184843f3401d0,2.70,221,85,39865.22,8227.31,920.98,0.44,0.72,-0.22,0.14,0.09,0.04,0.59,0.11,2025-01-28 14:20:00+00:00,1.11,1141.37,0.14,208.51,0.23,0.21,0.11,0.12,0.71,0.22,-0.02,0.02,0.23,0.71,0.27,0.27
21,0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,0.89,1719,665,51350.32,7340.58,903.36,0.23,0.35,-0.18,0.06,0.03,0.01,0.59,0.05,2026-01-27 23:20:00+00:00,0.30,482.24,0.07,372.72,0.41,0.14,0.12,0.04,0.98,0.18,-0.14,0.02,0.41,0.31,0.04,0.04
19,0x6b61b495ee20b5a161b7f75649e3b8888b217a33,3.52,149,25,11271.70,6048.95,783.70,0.46,0.63,-0.07,0.27,0.24,0.13,0.87,0.44,2026-01-26 12:20:00+00:00,0.47,190.05,0.03,33.62,0.04,0.54,0.13,0.06,0.66,0.07,-1.05,0.07,0.04,1.85,0.11,0.11


In [373]:
df_slice.head(1)

,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional
0,0x00090e8b4fa8f88dc9c1740e460dd0f670021d43,0x09b3e4c716df2579afa0ede1bf00edd84f8cbe1e335785f90336e7a909c52f8e,2025-09-29 16:48:21+00:00,BUY,159.75,159.75,0.68,108.63,159.75,51.12,0.00,True,1.00,0x135f651765de9f28e983b907579bae719e65327ff2aa34bbafd61133a908fe2b,1,True,-0.00,0.00,0.00,2025-10-01T00:00:00Z,US government shutdown by October 1?,"[Politics, Trump, Congress, funding, Trump Presidency, house, Gov Shutdown, 2025 Predictions, Ec...",Politics,98045093822431547938365423095786421804452886266888528428746910959667685407691,Yes,51.12,108.63


In [374]:
wallets = set(wallet_cohorts['multi_filter']['wallet'])

df = df_full[~df_full["is_train"]].copy()
print(len(df))

df_wallets = df[
    (df['wallet'].isin(wallets))
    & ~df['is_train']
    ].copy()
print(len(df_wallets))

df = df_wallets.groupby('condition_id').agg(
    pnl=('pnl', 'sum'),
    notional=('notional', 'sum'),
    sell_count=('side', lambda x: (x == 'SELL').sum()),
    buy_count=('side', lambda x: (x == 'BUY').sum()),
    wallets=('wallet', lambda x: x.nunique()),
).sort_values(by="pnl", key=np.abs, ascending=False).merge(mdf, on="condition_id", how="left")

3905163
6571


In [375]:
MARKET = '0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20'

market_def = mdf.loc[mdf['condition_id'] == MARKET].head(1).squeeze()
print(f"Market: {market_def}")

dfc = df_full[(df_full['condition_id'] == MARKET)
              & (df_full['wallet'].isin(wallets))].copy()
# dfc['bucket'] = dfc['dt'].dt.floor('1h')


dfc = dfc.groupby(['dt', 'wallet', 'side', 'outcome']).agg( 
        pnl=('pnl', 'sum'),
        notional=('notional', 'sum'),
        quantity=('quantity', 'sum'),
        position=('position', 'last'),
        avg_price=('price', lambda x: (x * dfc.loc[x.index, 'quantity']).sum() / dfc.loc[x.index, 'quantity'].sum()),
        copyable_qty=('copyable_qty', 'sum'),
        copyable_pnl=('copyable_pnl', 'sum'),
)[['pnl', 'position', 'notional', 'quantity', 'avg_price', 'copyable_qty', 'copyable_pnl']].reset_index().sort_values(['dt', 'wallet', 'side', 'outcome'])

Market: condition_id                   0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20
end_date_iso                                                                 2026-01-31T00:00:00Z
question                                                    US strikes Iran by February 28, 2026?
tags                [Politics, Iran, Middle East, Geopolitics, World, Parent For Derivative, HFC]
primary_tag                                                                              Politics
winner_token_id    110790003121442365126855864076707686014650523258783405996925622264696084778807
outcome                                                                                       Yes
Name: 59764, dtype: object


In [376]:
# ── positions per wallet + sum, dual-axis (position + price) ──────────────────
import plotly.graph_objects as go
from plotly.subplots import make_subplots

if len(dfc) != 0:

    # ── 1. build per-bucket price series for YES (volume-weighted avg)
    _yes = dfc[dfc['outcome'] == 'Yes'].copy()
    _yes['_wv'] = _yes['avg_price'] * _yes['quantity']
    price_ts = (
        _yes.groupby('dt')[['_wv', 'quantity']].sum()
        .rename(columns={'quantity': '_qty'})
        .assign(vwap=lambda d: d['_wv'] / d['_qty'])[['vwap']]
        .reset_index()
        .sort_values('dt')
    )

    # ── 2. net YES position per wallet per timestamp (YES pos - NO pos)
    # position is cumulative after each trade; take last value per (dt, wallet, outcome)
    pos_per_wallet = (
        dfc
        .groupby(['dt', 'wallet', 'outcome'])['position']
        .last()
        .reset_index()
    )
    pos_yes = pos_per_wallet[pos_per_wallet['outcome'] == 'Yes'].rename(columns={'position': 'pos_yes'})[['dt', 'wallet', 'pos_yes']]
    pos_no = pos_per_wallet[pos_per_wallet['outcome'] == 'No'].rename(columns={'position': 'pos_no'})[['dt', 'wallet', 'pos_no']]
    net_pos = (
        pos_yes
        .merge(pos_no, on=['dt', 'wallet'], how='outer')
        .fillna(0)
    )
    net_pos['net'] = net_pos['pos_yes'] - net_pos['pos_no']
    net_pos = net_pos.sort_values(['wallet', 'dt']).reset_index(drop=True)

    # ── 3. weight wallets by proper probabilistic score (Brier skill) with shrinkage
    wallet_scores = (
        train_b_metrics[['wallet', 'brier_skill', 'open_buy_trades']]
        .drop_duplicates('wallet')
        .copy()
    )
    wallet_scores['score'] = wallet_scores['brier_skill'].clip(lower=0.0).fillna(0.0)
    wallet_scores['confidence'] = (
        wallet_scores['open_buy_trades'].fillna(0.0)
        / (wallet_scores['open_buy_trades'].fillna(0.0) + 25.0)
    )
    wallet_scores['wallet_weight'] = wallet_scores['score'] * wallet_scores['confidence']
    wallet_weights = wallet_scores.set_index('wallet')['wallet_weight']

    # ── 4. normalize by each wallet's typical move size in this market
    net_pos['net_step'] = net_pos.groupby('wallet')['net'].diff()
    typical_move = net_pos.groupby('wallet')['net_step'].apply(
        lambda s: s.abs().dropna().median() if s.notna().any() else np.nan
    ).rename('typical_move')
    positive_moves = typical_move[typical_move > 0]
    fallback_move = positive_moves.median() if not positive_moves.empty else 1.0
    typical_move = typical_move.fillna(fallback_move).clip(lower=fallback_move * 0.25)

    # ── 5. carry positions forward so the signal reflects current wallet stance
    timeline = price_ts['dt'].sort_values().drop_duplicates()
    net_panel = (
        net_pos.pivot(index='dt', columns='wallet', values='net')
        .sort_index()
        .reindex(timeline)
        .ffill()
        .fillna(0.0)
    )
    available_wallets = net_panel.columns.intersection(wallet_weights.index).intersection(typical_move.index)
    net_panel = net_panel[available_wallets]
    wallet_weights = wallet_weights.reindex(available_wallets).fillna(0.0)
    typical_move = typical_move.reindex(available_wallets)
    normalized_panel = net_panel.divide(typical_move, axis=1)
    weighted_panel = normalized_panel.multiply(wallet_weights, axis=1)

    signal_ts = pd.DataFrame({
        'dt': net_panel.index,
        'total_net': net_panel.sum(axis=1).to_numpy(),
        'weighted_position': weighted_panel.sum(axis=1).to_numpy(),
    })

    fig = make_subplots(
        rows=1, cols=1,
        subplot_titles=['YES token (positive weighted signal => predicts YES)'],
        specs=[[{'secondary_y': True}]],
    )

    COLORS = [
        '#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66',
        '#77BEDB', '#F7A35C', '#90ED7D', '#8085E9', '#F15C80',
    ]

    top_wallets = net_panel.abs().max().sort_values(ascending=False).head(10).index

    # ── per-wallet net position (step: hold latest value, no interpolation)
    for c_idx, wallet in enumerate(top_wallets):
        wdf = (
            net_panel[[wallet]].reset_index()
            .rename(columns={wallet: 'net'})
            .sort_values('dt')
        )
        fig.add_trace(
            go.Scatter(
                x=wdf['dt'],
                y=wdf['net'],
                mode='lines',
                name=f'{wallet[:8]}...',
                line=dict(color=COLORS[c_idx % len(COLORS)], width=1, shape='hv'),
                legendgroup=wallet,
                opacity=0.6,
            ),
            row=1, col=1, secondary_y=False,
        )

    # ── total net position line (primary y, bold)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['total_net'],
            mode='lines',
            name='SUM (net YES)',
            line=dict(color='#222222', width=3, shape='hv'),
            legendgroup='sum_net',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── weighted normalized wallet position (primary y)
    fig.add_trace(
        go.Scatter(
            x=signal_ts['dt'],
            y=signal_ts['weighted_position'],
            mode='lines',
            name='Weighted normalized position',
            line=dict(color='#C44E52', width=3, shape='hv'),
            legendgroup='weighted_signal',
        ),
        row=1, col=1, secondary_y=False,
    )

    # ── YES price line (secondary y)
    fig.add_trace(
        go.Scatter(
            x=price_ts['dt'],
            y=price_ts['vwap'],
            mode='lines+markers',
            name='Price (YES)',
            line=dict(color='#888888', width=2, dash='dot', shape='hv'),
            marker=dict(size=4),
            legendgroup='price_yes',
        ),
        row=1, col=1, secondary_y=True,
    )

    fig.add_hline(y=0, line=dict(color='#BBBBBB', width=1, dash='dash'))
    fig.update_yaxes(title_text='Net position / weighted normalized total', row=1, col=1, secondary_y=False)
    fig.update_yaxes(title_text='Price (USDC)', row=1, col=1, secondary_y=True, range=[0, 1])

    fig.update_layout(
        title=f'Wallet net YES positions and weighted signal - {MARKET[:16]}...',
        height=500,
        template='plotly_white',
        legend=dict(orientation='v', x=1.05),
    )
    fig.show(renderer='browser')


In [377]:
# dominant tags
wallet_fills = df_full[df_full['wallet'].isin(wallet_cohorts['multi_filter']['wallet'])]

dominant_tags = (
    wallet_fills
    .groupby(['wallet', 'primary_tag'], as_index=False)[['quantity', 'pnl']].sum()
    .sort_values(['wallet', 'quantity'], ascending=[True, False])
    .assign(total_qty=lambda df: df.groupby('wallet')['quantity'].transform('sum'))
    .drop_duplicates('wallet')
    .assign(primary_tag_ratio=lambda df: df['quantity'] / df['total_qty'])
    .rename(columns={
        'quantity': 'primary_tag_qty'
    })[['wallet', 'primary_tag', 'primary_tag_qty', 'primary_tag_ratio', 'pnl']]
)

In [378]:
print(len(dominant_tags))
dominant_tags.sort_values('pnl', ascending=False).head(20)

22


,wallet,primary_tag,primary_tag_qty,primary_tag_ratio,pnl
194,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,Politics,291755.39,0.80,54998.55
18,0x0cb10c40b0776e9ee8cef970af85724654dda76c,Politics,726552.02,0.86,39052.74
92,0x22dbd51e1a4e10fff072fa24300238c64033190f,Politics,1021469.54,0.66,29775.30
130,0x2853240a0f4e9e11a949a5cfa6e0fe953a293482,Politics,292981.48,0.75,20971.10
48,0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,Politics,139836.63,0.63,11857.87
151,0x444e1b2ccf56b63e6b54e1357df184843f3401d0,Politics,56926.80,0.77,8333.11
222,0x9b2fcb79bdb8a6a6174da327e9335f7889c6d8f8,Politics,130418.06,0.93,7478.90
190,0x6b61b495ee20b5a161b7f75649e3b8888b217a33,Politics,18549.80,1.00,6048.95
185,0x672225e5e1aba1c970ac613efd1505f4b7a10762,Finance,358855.95,0.87,5750.05
262,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,Politics,54937.20,0.67,4358.33


In [379]:
# from polymarket_analysis.data.tags import load_tag_map
# from polymarket_analysis.data.tags import dominant_tag_stats

# split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

# _result = dominant_tag_stats(
#     df_trades=df_full[df_full['dt'] >= split_dt],
#     wallets=wallet_cohorts["multi_filter"]["wallet"],
# )

# print(f"Wallets: {len(_result)}")
# _result[_result['primary_tag'] == 'Politics'].head(5)


In [380]:
# _result.groupby('primary_tag').agg(
#     tag_pnl=('tag_pnl', 'sum'),
#     wallets=('wallet', 'nunique')
#     )

In [381]:
watched_wallets = wallet_cohorts['multi_filter'].sort_values("copyable_pnl", key=abs, ascending=False)['wallet'].head(10).to_list()
for w in watched_wallets:
    wallet_df = wallet_vol_train[wallet_vol_train['wallet'] == w]
    wallet_cohorts[w] = wallet_df.copy().reset_index(drop=True)

In [382]:
print(len(wallet_cohorts['multi_filter']))
pd.set_option('display.max_rows', 100)
wallet_cohorts["multi_filter"].sort_values("final_score", ascending=False).head(30)


22


,wallet,pnl_volatility,num_buckets,num_markets,total_notional,total_pnl,copyable_pnl,top5_pnl_pct,top10_pnl_pct,worst5_pnl_pct,top_market_pnl_pct,top_market_abs_pnl_pct,market_pnl_hhi,positive_bucket_share,median_roi,median_dt,average_roi,max_drawdown,max_drawdown_to_pnl,max_copyable_drawdown,max_copyable_drawdown_to_copyable_pnl,return,copyable_pnl_factor,copyable_roi,sample_mult,downside_tail,predictability_score,copyable_efficiency,copyable_dd_ratio,copyable_score,final_score,wallet_quality
0,0x815faf650a093427c1386f3fc7465d03bdc7f1b3,1.06,247,82,358092.33,1596.22,459.84,0.50,0.73,-0.20,0.27,0.25,0.11,0.78,0.00,2025-02-03 14:20:00+00:00,3.76,141.35,0.09,99.50,0.22,0.00,0.29,1.08,0.73,0.20,5.94,0.00,0.22,2.03,4.38,4.38
1,0x6bc74c392c320cfe10d5be61db978a58c8444ad4,3.66,1287,195,86665.60,42409.16,6837.74,0.22,0.38,-0.23,0.31,0.18,0.06,0.55,0.11,2026-02-13 11:15:00+00:00,3.40,7373.80,0.17,5604.27,0.82,0.49,0.16,0.55,0.94,0.23,4.32,0.08,0.82,2.41,3.55,3.55
2,0x3e438fdb437d90925b47b6f8ab85405ba8ffa83f,0.99,56,34,5978.58,1037.16,644.30,0.54,0.79,-0.27,0.14,0.10,0.05,0.72,0.18,2025-06-24 10:30:00+00:00,0.91,107.46,0.10,73.91,0.11,0.17,0.62,0.57,0.53,0.27,0.77,0.11,0.11,4.31,2.19,2.19
3,0xe97b7b1396a99341b32984fa8bbf62f2f68ab2ea,1.36,221,47,50819.14,6107.90,1358.21,0.40,0.62,-0.21,0.14,0.12,0.07,0.77,0.12,2025-02-23 16:25:00+00:00,1.07,631.30,0.10,449.21,0.33,0.12,0.22,0.24,0.71,0.21,1.03,0.03,0.33,1.04,1.04,1.04
4,0x0cb10c40b0776e9ee8cef970af85724654dda76c,1.04,5472,1784,299213.40,46376.75,6731.76,0.14,0.22,-0.06,0.06,0.03,0.00,0.51,0.01,2025-11-07 17:15:00+00:00,0.89,1030.77,0.02,1082.54,0.16,0.15,0.15,0.13,1.00,0.06,1.13,0.02,0.16,0.83,1.01,1.01
5,0x672225e5e1aba1c970ac613efd1505f4b7a10762,1.46,1113,319,255194.40,4860.34,1381.57,0.42,0.56,-0.03,0.13,0.13,0.05,0.88,0.00,2026-01-28 21:25:00+00:00,0.81,160.00,0.03,100.36,0.07,0.02,0.28,0.23,0.92,0.03,0.77,0.01,0.07,0.77,0.77,0.77
6,0x5042e6dc8a612c493881a3e67519cc09f5f4fcb0,0.59,1647,658,39055.76,2022.02,256.77,0.27,0.44,-0.28,0.07,0.06,0.01,0.65,0.05,2025-10-29 18:50:00+00:00,0.74,407.05,0.20,74.49,0.29,0.05,0.13,0.09,0.97,0.28,1.05,0.01,0.29,0.22,0.72,0.72
7,0x0dedae6a02ea2ff8018ba5f277632919ed1c9025,0.52,3178,1105,82151.41,13233.71,486.75,0.19,0.29,-0.09,0.06,0.03,0.00,0.55,0.03,2025-09-30 18:15:00+00:00,0.80,613.32,0.05,404.04,0.83,0.16,0.04,0.03,1.00,0.09,1.45,0.01,0.83,-0.43,0.70,0.70
8,0x9783178832982d420baa733d2ec29b020eb9264f,0.77,53,26,8227.49,465.19,380.91,0.50,0.72,-0.14,0.16,0.15,0.08,0.95,0.04,2026-01-18 10:00:00+00:00,0.04,58.13,0.12,19.62,0.05,0.06,0.82,0.03,0.52,0.14,-0.25,0.05,0.05,2.07,0.68,0.68
9,0xb78ddd792ad7d1f603fcd046c662d6d9adbf835d,1.52,124,67,13154.70,2148.21,1169.65,0.43,0.68,-0.34,0.16,0.08,0.03,0.53,0.06,2026-01-26 13:37:30+00:00,0.15,250.83,0.12,107.42,0.09,0.16,0.54,0.08,0.64,0.34,-0.89,0.09,0.09,2.92,0.63,0.63


In [383]:
selected_wallets = wallet_cohorts["multi_filter"].copy()
selected_wallets.to_parquet(WORKSPACE_DIR / "selected_wallets_v2.parquet", index=False)
print(f"Saved selected wallets: {len(selected_wallets):,}")


Saved selected wallets: 22


In [384]:
WORKSPACE_DIR / "selected_wallets_v2.parquet"


PosixPath('../../data/trade_signals_workspace_v2/selected_wallets_v2.parquet')

## Build and save strategy registry

All cohorts × all trigger variants → persisted `WalletSelectionStrategy` files.
The backtest loads these directly.

In [385]:
from polymarket_analysis.strategy.definition import TriggerSpec, WalletSelectionStrategy
from polymarket_analysis.strategy.registry import save_strategy

SIGNAL_THRESHOLD = 0.65

selected_wallets = wallet_cohorts['multi_filter'].copy().reset_index(drop=True)
multi_filter_wallets = selected_wallets.copy()
if 'wallet_quality' not in multi_filter_wallets.columns:
    if 'final_score' in multi_filter_wallets.columns:
        multi_filter_wallets['wallet_quality'] = multi_filter_wallets['final_score']
    else:
        multi_filter_wallets['wallet_quality'] = 0.0

strategy_specs = [
    (
        'multi_filter__score_threshold',
        f'multi_filter | score >= {SIGNAL_THRESHOLD:.2f} (Kelly)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.score_threshold',
            params={'threshold': SIGNAL_THRESHOLD, 'dynamic_sizing': True},
            mode='frame',
        ),
    ),
    (
        'multi_filter__all_open_buys',
        'multi_filter | all open-buys (fixed stake)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.all_open_buys',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
    (
        'multi_filter__copy_triggers',
        'multi_filter | copy all triggers (tight slippage)',
        TriggerSpec(
            fn_ref='polymarket_analysis.strategy.triggers.copy_triggers',
            params={'dynamic_sizing': False},
            mode='frame',
        ),
    ),
]

all_strategies = []
for strategy_id, name, trigger in strategy_specs:
    all_strategies.append(
        WalletSelectionStrategy(
            strategy_id=strategy_id,
            name=name,
            selection_method='multi_filter',
            trigger=trigger,
            wallets=multi_filter_wallets.copy(),
            params={
                'selection_metric': 'final_score',
                'top_n': len(multi_filter_wallets),
                'signal_threshold': SIGNAL_THRESHOLD if trigger.fn_ref.endswith('score_threshold') else None,
            },
            metadata={
                'cohort': 'multi_filter',
                'selection_metric': 'final_score',
                'signal_threshold': SIGNAL_THRESHOLD,
                'top_n': len(multi_filter_wallets),
                'end_date_train': str(END_DATE_TRAIN),
                'train_a_end_date': str(TRAIN_A_END_DATE),
                'selection_notes': 'full-stream copy-trigger ranking with absolute predictability/copyable scores',
            },
        )
    )

for strategy in all_strategies:
    parquet_path, json_path = save_strategy(strategy, WORKSPACE_DIR)
    print(f"Saved [{strategy.strategy_id}] wallets={len(strategy.wallets):3d} trigger={strategy.trigger.fn_ref.split('.')[-1]}")

print(f'\nTotal strategies saved: {len(all_strategies)}')


Saved [multi_filter__score_threshold] wallets= 22 trigger=score_threshold
Saved [multi_filter__all_open_buys] wallets= 22 trigger=all_open_buys
Saved [multi_filter__copy_triggers] wallets= 22 trigger=copy_triggers

Total strategies saved: 3


## Strategy registry summary

In [386]:
from polymarket_analysis.strategy.registry import load_all_strategies

registry = load_all_strategies(WORKSPACE_DIR)
summary_rows = []
for sid, s in registry.items():
    summary_rows.append({
        'strategy_id': s.strategy_id,
        'name': s.name,
        'selection_method': s.selection_method,
        'num_wallets': len(s.wallets),
        'trigger_fn': s.trigger.fn_ref.split('.')[-1],
        'threshold': s.trigger.params.get('threshold'),
        'dynamic_sizing': s.trigger.params.get('dynamic_sizing'),
    })

pd.DataFrame(summary_rows).sort_values('strategy_id').reset_index(drop=True)


,strategy_id,name,selection_method,num_wallets,trigger_fn,threshold,dynamic_sizing
0,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__all_open_buys,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | all open-buys (fixed stake),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,all_open_buys,NaN,False
1,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__copy_triggers,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | copy all triggers (tight slippage),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,copy_triggers,NaN,False
2,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf__score_threshold,0x0c058138ee8749433656cb5c696e0fd3dc7dfabf | score >= 0.65 (Kelly),0x0c058138ee8749433656cb5c696e0fd3dc7dfabf,1,score_threshold,0.65,True
3,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__all_open_buys,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | all open-buys (fixed stake),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,all_open_buys,NaN,False
4,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__copy_triggers,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | copy all triggers (tight slippage),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,copy_triggers,NaN,False
5,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9__score_threshold,0x17e9a22d1733f26f18af3157864c1da0e5eafcc9 | score >= 0.65 (Kelly),0x17e9a22d1733f26f18af3157864c1da0e5eafcc9,1,score_threshold,0.65,True
6,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__all_open_buys,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | all open-buys (fixed stake),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,all_open_buys,NaN,False
7,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__copy_triggers,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | copy all triggers (tight slippage),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,copy_triggers,NaN,False
8,0x480e1e120dba94077cb8571fa72f6bc58c7971e9__score_threshold,0x480e1e120dba94077cb8571fa72f6bc58c7971e9 | score >= 0.65 (Kelly),0x480e1e120dba94077cb8571fa72f6bc58c7971e9,1,score_threshold,0.65,True
9,0x64e86fefd50cdb6259eb00ff146d9aac03497819__all_open_buys,0x64e86fefd50cdb6259eb00ff146d9aac03497819 | all open-buys (fixed stake),0x64e86fefd50cdb6259eb00ff146d9aac03497819,1,all_open_buys,NaN,False


## Wallet PnL over time

Three independent plots:
- **Training plot** — cohort aggregate cumulative PnL over the training period only
- **Test plot** — cohort aggregate cumulative PnL over the test period only (starts from 0)
- **Individual plot** — per-wallet cumulative PnL spanning train + test, with a train/test split
  vline and wallet address labels; test portion resets to 0 at the split boundary

Set `PLOT_WALLET_PNL = False` to skip (e.g. when running headlessly).


In [387]:
PLOT_WALLET_PNL  = True
TOP_N_INDIVIDUAL = 10   # wallets shown in panel 1 per cohort


In [388]:
df_fills = df_full.copy()
df_fills['copyable_pnl_exposure'] = np.where( 
    df_fills['copyable_qty'] > 0,
    df_fills['price'] * df_fills['copyable_qty'],
    np.nan
)

# Normalise grouped schema → canonical fill-level names
if "total_quantity" in df_fills.columns and "quantity" not in df_fills.columns:
    df_fills = df_fills.rename(columns={
        "total_quantity": "quantity",
        "avg_price": "price",
        "trade_value_usdc": "usdc_amount",
    })

df_fills["usdc_amount"]      = df_fills["usdc_amount"].astype(float)
df_fills["final_value_usdc"] = df_fills["final_value_usdc"].astype(float)
df_fills["quantity"]         = df_fills["quantity"].astype(float)

split_dt = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

len(df_fills), len(df_fills[df_fills["dt"] < split_dt]), len(df_fills[df_fills["dt"] >= split_dt])

(17649785, 13744622, 3905163)

In [389]:
markets = df_fills[(df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))][['condition_id', 'tags', 'primary_tag']]
markets = markets.groupby('condition_id').agg(
    tags=('tags', 'first'),
    primary_tag=('primary_tag', 'first'),
).reset_index()
markets['has_mentions'] = markets['tags'].apply(lambda tags: 'Mentions' in tags)
mention_markets = set(markets[markets['has_mentions']]['condition_id'])
del markets
len(mention_markets)

4762

In [390]:
filter_fills = df_fills[
    (df_fills['wallet'].isin(wallet_cohorts['multi_filter']['wallet']))
    & (df_fills['side'] == 'BUY')
    & (df_fills['primary_tag'] == 'Politics')
    & (~df_fills['condition_id'].isin(mention_markets))
    & (df_fills["dt"] >= split_dt)
    ].copy().reset_index(drop=True)

print(len(filter_fills))
filter_fills = filter_fills[filter_fills['copyable_qty'] >= 1].copy().reset_index(drop=True)
print(len(filter_fills))
filter_fills.head(2)

1482
798


,wallet,condition_id,dt,side,position,quantity,price,usdc_amount,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count,end_date_iso,question,tags,primary_tag,winner_token_id,outcome,pnl,notional,copyable_pnl_exposure
0,0x0cb10c40b0776e9ee8cef970af85724654dda76c,0x0eee746a50b8852348403c3832b0c1fde83ed7604953e3f82b0a01e19e1c3fd5,2026-05-28 03:16:00+00:00,BUY,1500.00,1500.00,0.00,4.50,0.00,-4.50,-4.50,False,0.00,0x3ca296c17e1fc3d3f5f5170dcc51881ea59072f8016498de1a78040b254e9cd9,1,False,1500.00,10871.55,4.00,2026-05-31T00:00:00Z,"Will Donald Trump dance on May 25, 2026?","[Politics, Trump, Culture, Trump Daily]",Politics,101701365191317131762612580843637826322216807284896284696153769493375151124770,Yes,-4.50,4.50,4.50
1,0x22dbd51e1a4e10fff072fa24300238c64033190f,0x084a4aac9da433824013ad192196193d7d1bd8293c0fedc9eb7e687909fd1d0f,2026-04-24 11:15:16+00:00,BUY,200.00,200.00,1.00,199.40,200.00,0.60,0.60,True,1.00,0x51d64507d6eab812758147d999469c9bfe21f4c41c787d29b3e8fd298f7743fc,1,False,200.00,32.65,7.00,2026-04-24T00:00:00Z,"US x Iran permanent peace deal by April 24, 2026?","[Politics, Iran, Trump, ceasefire, Geopolitics, Iran Ceasefire, U.S. x Iran, Agreement, 10-point]",Politics,81526178641185308468785014715929124534877997252766637103131165456634740242354,No,0.60,199.40,199.40


In [391]:
pd.set_option('display.max_colwidth', 100)

In [392]:
filter_fills["bucket"] = filter_fills["dt"].dt.floor('5min')

MAX_EXPOSURE = 100

df = filter_fills.groupby(['bucket', 'condition_id', 'wallet', 'side']).agg(
    copyable_pnl_exposure=('copyable_pnl_exposure', 'sum'),
    total_copyable_qty=('copyable_qty', 'sum'),
    trade_pnl =('trade_pnl', 'sum'),
    copyable_pnl = ('copyable_pnl', 'sum'),
    trades=('condition_id', 'count'),
    copyable_qty=('copyable_qty', 'sum'),
    wallets=('wallet', lambda x: x.nunique()),
).reset_index()

scale = np.minimum(1, MAX_EXPOSURE / df["copyable_pnl_exposure"].replace(0, np.nan))

df = df.assign(
    scaled_copyable_pnl_exposure = df["copyable_pnl_exposure"] * scale,
    scaled_copyable_pnl = df["copyable_pnl"] * scale,
    scaled_copyable_qty = df["copyable_qty"] * scale,
)

df.head(10)

,bucket,condition_id,wallet,side,copyable_pnl_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,copyable_qty,wallets,scaled_copyable_pnl_exposure,scaled_copyable_pnl,scaled_copyable_qty
0,2026-04-16 01:50:00+00:00,0x6d64853f670b85151d2441b2bdea142f2031b573947ab2887e82585546fdc7f4,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,35.97,42.32,30.00,6.35,1,42.32,1,35.97,6.35,42.32
1,2026-04-16 02:00:00+00:00,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,17.00,17.05,0.05,0.05,1,17.05,1,17.00,0.05,17.05
2,2026-04-16 02:05:00+00:00,0x531afb8dafa0c026464827a4d6c4260743f6d2908101ae43645d6513e6bc5438,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,166.18,166.68,0.50,0.50,1,166.68,1,100.00,0.30,100.30
3,2026-04-16 02:20:00+00:00,0x136f5a0c27a62cf9a2e40a4f48425e43d61b9571a53a2529372c0065f3218a73,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,54.56,56.60,50.23,2.05,3,56.60,1,54.56,2.05,56.60
4,2026-04-16 11:10:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,124.00,200.00,76.00,76.00,1,200.00,1,100.00,61.29,161.29
5,2026-04-16 11:55:00+00:00,0x136f5a0c27a62cf9a2e40a4f48425e43d61b9571a53a2529372c0065f3218a73,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,462.99,500.00,37.01,37.01,2,500.00,1,100.00,7.99,107.99
6,2026-04-16 14:30:00+00:00,0xcd26bdb63b81dfcc5482c42e9372c8a46091b1de1123d2afefb83df83fe859fd,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,84.00,1200.00,-84.00,-84.00,1,1200.00,1,84.00,-84.00,1200.00
7,2026-04-16 17:00:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,201.00,300.00,99.00,99.00,1,300.00,1,100.00,49.25,149.25
8,2026-04-16 17:50:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x22dbd51e1a4e10fff072fa24300238c64033190f,BUY,65.00,100.00,35.00,35.00,1,100.00,1,65.00,35.00,100.00
9,2026-04-16 17:55:00+00:00,0xceb6dfaa2cf5abc9d47ebc867b984a7715104944249274e8a483a2e17473e5f5,0x672225e5e1aba1c970ac613efd1505f4b7a10762,BUY,80.00,200.00,-80.00,-80.00,2,200.00,1,80.00,-80.00,200.00


In [393]:
df = (df.merge(mdf, on='condition_id', how='inner')
    .sort_values(['bucket', 'copyable_pnl_exposure'], ascending=[True, False])
    .reset_index(drop=True)
)
df['end_date_iso'] = pd.to_datetime(df['end_date_iso'], utc=True)
df = df[df['end_date_iso'] > split_dt].copy().reset_index(drop=True)

In [394]:
df_plot = df.groupby('end_date_iso').agg(
    scaled_copyable_pnl=('scaled_copyable_pnl', 'sum'),
    ending_exposure=('scaled_copyable_pnl_exposure', 'sum'),
).reset_index()

df.sort_values('bucket', inplace=True)
df['scaled_copyable_pnl_cumsum'] = df['scaled_copyable_pnl'].cumsum()

df_pnl = df[['bucket', 'scaled_copyable_pnl_cumsum']].copy()

# exposure ends at EOD of end_date, so shift by 24h
df_exposure = df_plot[['end_date_iso', 'ending_exposure']].copy()
df_exposure['end_date_iso'] = pd.to_datetime(df_exposure['end_date_iso']) + pd.Timedelta(days=1)
df_exposure.rename(columns={'end_date_iso': 'dt', 'ending_exposure': 'exposure_delta'}, inplace=True)

resolution_exposure = df_exposure.groupby('dt').agg(exposure_delta=('exposure_delta', 'sum')).reset_index()
resolution_exposure['exposure_delta'] = resolution_exposure['exposure_delta'] * -1

buy_exposure = df[['bucket', 'scaled_copyable_pnl_exposure']].rename(columns={'bucket': 'dt', 'scaled_copyable_pnl_exposure': 'exposure_delta'})

df_exposure = ( pd.concat([resolution_exposure, buy_exposure], ignore_index=True)
    .reset_index(drop=True)
)

df_exposure.sort_values('dt', inplace=True)
df_exposure['exposure'] = df_exposure['exposure_delta'].cumsum()

# df_plot['end_date_iso'] = pd.to_datetime(df_plot['end_date_iso']) + pd.Timedelta(days=1)

import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df_exposure['dt'],
    y=df_exposure['exposure'],
    mode='lines',
    name='exposure'
))

fig.add_trace(go.Scatter(
    x=df_pnl['bucket'],
    y=df_pnl['scaled_copyable_pnl_cumsum'],
    mode='lines',
    name='scaled_copyable_pnl_cumsum'
))

fig.show()

In [395]:
import plotly.express as px

# Aggregate by bucket
bucket_pnl = df.groupby('bucket')['scaled_copyable_pnl'].sum().reset_index()
bucket_pnl['scaled_copyable_pnl'] = bucket_pnl['scaled_copyable_pnl'].cumsum()

# Plot with Plotly
fig = px.line(
    bucket_pnl,
    x='bucket',
    y='scaled_copyable_pnl',
    title='Total Scaled Copyable PnL by Time Bucket',
    labels={'bucket': 'Time Bucket', 'scaled_copyable_pnl': 'Scaled Copyable PnL'}
)
fig.show()

In [396]:
( 
    df.groupby('condition_id').agg(
        total_copyable_exposure=('copyable_pnl_exposure', 'sum'),
        total_copyable_qty=('copyable_qty', 'sum'),
        trade_pnl =('trade_pnl', 'sum'),
        copyable_pnl = ('copyable_pnl', 'sum'),
        trades=('condition_id', 'count'),
        wallets=('wallet', lambda x: x.nunique()),
 ).sort_values('copyable_pnl', ascending=False).head(10)
)

,total_copyable_exposure,total_copyable_qty,trade_pnl,copyable_pnl,trades,wallets
condition_id,,,,,,
0x4ba348328e4d4ddee9e6734c9a369b2e8138611651f9f6dc8f59dea51df6c734,764.10,3340.63,2659.95,2503.89,15,2
0x807ba575f84169bdb061dd05da3e8eb314a129aa152e31518c2aff9979d5f986,985.38,1702.01,952.72,716.63,9,1
0xe208a07bd71069b04e3b89de92f71da8b155d45dc2e527dc46c3ceb2a63b307b,365.04,872.96,529.01,507.92,6,2
0x61dedda02f8abb229d3c413e6ab4861446d075807e12532f23642c1d82839b9a,2886.14,3275.75,630.48,389.61,20,1
0x71c49500d051a9b32b9aa5bc22e0a6925a8b2282840c3e62e0a3afcd0403ba03,2120.27,2587.43,603.19,386.91,11,2
0x2135ffcb43ba9103bb6acf7116d2d5aa98bef6d9eb3dc9c85ea00cb79513f3ec,1700.10,2817.48,359.01,316.90,14,3
0xa743ce4a2a42aa232c2caa41aa27837ff1cde07781a1e9a726342065c3d86ba6,73.85,1741.15,298.09,267.90,5,3
0x9ec068d68055cc7e14d82b67da6f8069423ce64d4cd3549aeb50116156a622f0,368.50,500.74,343.78,132.24,10,1
0x9967297556e4fd7f7e1e688a2e44d6fdbc42b0700576a1e4e67aa37764ac640a,137.55,262.92,144.45,125.37,3,2


In [397]:
from polymarket_analysis.visualization.wallet_plots import (
    plot_wallet_selection_pnl,
    plot_wallet_individual_pnl,
)

if PLOT_WALLET_PNL:
    # Cohort aggregate PnL — training period
    fig_train = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="train",
        title="Wallet selection cohorts — cohort cumulative PnL (training period)",
    )
    fig_train.show(renderer="browser")

    # Cohort aggregate PnL — test period (starts from 0)
    fig_test = plot_wallet_selection_pnl(
        df_fills,
        wallet_cohorts,
        split_date=split_dt,
        period="test",
        title="Wallet selection cohorts — cohort cumulative PnL (test period)",
    )
    fig_test.show(renderer="browser")

    # # Individual wallet lines (train + test) with split vline and labels
    # fig_ind = plot_wallet_individual_pnl(
    #     df_fills,
    #     wallet_cohorts,
    #     split_date=split_dt,
    #     top_n_individual=TOP_N_INDIVIDUAL,
    #     title="Individual wallet cumulative PnL (train + test, resets at split)",
    # )
    # fig_ind.show(renderer="browser")

else:
    print("PLOT_WALLET_PNL=False — skipping plots")